# Stock Image Scraper 📸

In [ ]:
# STEP 1:- Importing the required libraries
import time
import requests
import pandas as pd
from tqdm import tqdm
import chromedriver_binary
from bs4 import BeautifulSoup
from selenium import webdriver
from openpyxl import Workbook
import re
import os
import urllib.request
from urllib.parse import urljoin
import shutil

In [2]:
# STEP :-2:- Setting up the Selenium WebDriver
driver = webdriver.Chrome()
driver.get("https://stock-pictures.netlify.app/")
time.sleep(5)  # Wait for the page to load

The chromedriver version (147.0.7727.57) detected in PATH at c:\Users\Nbinary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (148.0.7778.181); currently, chromedriver 148.0.7778.178 is recommended for chrome 148.*, so it is advised to delete the driver in PATH and retry


In [4]:
# STEP 3:- Scrapping the url
soup = BeautifulSoup(driver.page_source, "html.parser")
image_elements = soup.select('img.source-img[src$=".jpg"]')
print(f"Found {len(image_elements)} images on the page.")

Found 196 images on the page.


In [5]:
# STEP 3:- Scraping the image details
# image url, name tags, likes and comments on each image
image_data = []
for sp in soup.find_all('div', class_='container'):
    img = sp.find('img')
    
    # 1. Safely check if img exists AND has a src attribute before proceeding
    if img and img.get('src') and 'gif' not in img.get('src'):
        link = img.get('src')
        
        # 2. Safely extract tags (Default to empty string if missing)
        tags_div = sp.find('div', class_='tags')
        tags_text = ""
        if tags_div:
            # Assuming the first 7 chars were something like "Tags: "
            # A safer way is replacing the specific word, or just matching words
            raw_tags = tags_div.text[7:].strip().split(' ')
            tags_text = ' '.join(list(set(raw_tags)))
        
        # 3. Safely extract likes and comments (Default to 0 if missing)
        likes, comments = 0, 0
        likes_comments_div = sp.find('div', class_='likes-comments')
        
        if likes_comments_div:
            spans = likes_comments_div.find_all('span')
            
            # re.search(r'\d+', text) finds the first sequence of numbers in a string
            if len(spans) > 0:
                likes_match = re.search(r'\d+', spans[0].text)
                likes = int(likes_match.group()) if likes_match else 0
                
            if len(spans) > 1:
                comments_match = re.search(r'\d+', spans[1].text)
                comments = int(comments_match.group()) if comments_match else 0
        
        image_data.append([link, tags_text, likes, comments])

In [6]:
# STEP 4: Make in to a dataframe
df = pd.DataFrame(image_data, columns=['Image URL', 'Tags', 'Likes', 'Comments'])

In [7]:
df

,Image URL,Tags,Likes,Comments
0,https://cdn.pixabay.com/photo/2022/03/06/05/30...,"Sky, Sky Clouds, Blue Atmosphere,",196,55
1,https://cdn.pixabay.com/photo/2022/04/07/11/45...,"Hummingbird Ornithology, Bird,",76,20
2,https://cdn.pixabay.com/photo/2022/02/28/15/28...,"Subtropical Rainbow, Sea, Rainfall,",282,106
3,https://cdn.pixabay.com/photo/2022/04/04/02/52...,"Cherry Blossoms, Japan, Road, Sakura",42,11
4,https://cdn.pixabay.com/photo/2022/04/09/18/06...,"Marguerite, Flower, Cape Plant",39,15
...,...,...,...,...
191,https://cdn.pixabay.com/photo/2022/03/31/14/16...,"Bee, Pollination, Entomology, Macro",57,41
192,https://cdn.pixabay.com/photo/2022/04/06/05/26...,"Meadow Aurora Butterfly,",25,16
193,https://cdn.pixabay.com/photo/2021/12/02/13/02...,Weihnachtskerzen 4,70,24
194,https://cdn.pixabay.com/photo/2022/04/06/14/27...,description a (optional) Add,40,33


In [8]:
# STEP 5: Close the Selenium WebDriver
driver.quit()

In [9]:
# STEP 6: Save the data to an Excel file
df.to_excel("stock_images_data.xlsx", index=False)

In [10]:
# STEP 7: Download the images to a local folder
# Create a folder to save the images
path=[]
os.makedirs("downloaded_images", exist_ok=True)
# Use a browser-like User-Agent to avoid being blocked by the server
download_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}
# Download each image
for index, row in tqdm(df.iterrows(), total=len(df)):
    image_url = row['Image URL']
    if image_url.startswith('/'):
        image_url = urljoin("https://stock-pictures.netlify.app/", image_url)
    image_name = f"image_{index + 1}.jpg"
    save_path = os.path.join("downloaded_images", image_name)
    path.append(save_path)
    try:
        response = requests.get(image_url, headers=download_headers, timeout=30, stream=True)
        response.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=10240):
                if chunk:
                    f.write(chunk)
        print(f"Downloaded: {image_name}")
    except Exception as e:
        print(f"Failed to download {image_name}: {image_url} : {e}")

  0%|          | 0/196 [00:00<?, ?it/s]

  1%|          | 1/196 [00:00<03:03,  1.06it/s]

Downloaded: image_1.jpg


  1%|          | 2/196 [00:02<03:18,  1.03s/it]

Downloaded: image_2.jpg


...

Downloaded: image_194.jpg


 99%|█████████▉| 195/196 [02:07<00:00,  1.70it/s]

Downloaded: image_195.jpg


100%|██████████| 196/196 [02:08<00:00,  1.53it/s]

Downloaded: image_196.jpg


In [11]:
# STEP 8: Update the Excel file with the local image paths
df['Local Image Path'] = path
df.to_excel("stock_images_data.xlsx", index=False)

In [12]:
df

,Image URL,Tags,Likes,Comments,Local Image Path
0,https://cdn.pixabay.com/photo/2022/03/06/05/30...,"Sky, Sky Clouds, Blue Atmosphere,",196,55,downloaded_images\image_1.jpg
1,https://cdn.pixabay.com/photo/2022/04/07/11/45...,"Hummingbird Ornithology, Bird,",76,20,downloaded_images\image_2.jpg
2,https://cdn.pixabay.com/photo/2022/02/28/15/28...,"Subtropical Rainbow, Sea, Rainfall,",282,106,downloaded_images\image_3.jpg
3,https://cdn.pixabay.com/photo/2022/04/04/02/52...,"Cherry Blossoms, Japan, Road, Sakura",42,11,downloaded_images\image_4.jpg
4,https://cdn.pixabay.com/photo/2022/04/09/18/06...,"Marguerite, Flower, Cape Plant",39,15,downloaded_images\image_5.jpg
...,...,...,...,...,...
191,https://cdn.pixabay.com/photo/2022/03/31/14/16...,"Bee, Pollination, Entomology, Macro",57,41,downloaded_images\image_192.jpg
192,https://cdn.pixabay.com/photo/2022/04/06/05/26...,"Meadow Aurora Butterfly,",25,16,downloaded_images\image_193.jpg
193,https://cdn.pixabay.com/photo/2021/12/02/13/02...,Weihnachtskerzen 4,70,24,downloaded_images\image_194.jpg
194,https://cdn.pixabay.com/photo/2022/04/06/14/27...,description a (optional) Add,40,33,downloaded_images\image_195.jpg


In [14]:
# STEP 9: Clone the dataset
df_new = df.copy()
# two columns path and tags are there reset of the columns are removed
df_new = df_new[['Local Image Path', 'Tags']]
df_new.to_excel("stock_images_data_cloned.xlsx", index=False)
df_new

In [26]:
# STEP 10: Make the differnet folder for unque tags
# Get unique tags
unique_tags = set()
for tags in df_new['Tags']:
    unique_tags.update(tags.split())

# make a main folder to store the images in different folders based on tags
os.makedirs("tagged_images", exist_ok=True)

# Function to sanitize tag names for folder creation
def sanitize_tag(tag):
    tag = tag.strip()
    # remove common prefix like "Tags - " if present
    tag = re.sub(r'(?i)^tags\s*[-:]\s*', '', tag)
    # replace invalid filename chars with underscore
    tag = re.sub(r'[<>:"/\\|?*\n\r\t]', '_', tag)
    # replace commas and sequences of whitespace with underscore
    tag = re.sub(r'[,\s]+', '_', tag)
    # remove trailing dots/underscores/spaces
    tag = tag.rstrip('._ ')
    if not tag:
        tag = 'untitled'
    # avoid reserved Windows names
    if re.fullmatch(r'(?i)(con|prn|aux|nul|com[1-9]|lpt[1-9])', tag):
        tag = f'_{tag}'
        return tag

    # Create folders for each unique tag
    for tag in unique_tags:
        safe_tag = sanitize_tag(tag)
        tag_folder = os.path.join("tagged_images", safe_tag)
        os.makedirs(tag_folder, exist_ok=True)